# March Mania · Women’s bracket-context features
**Milestone 09 · a different source of information, two candidate features, one seed-status control.**

Round 08’s nonlinear possession features failed the fixed primary comparison. This notebook preserves that conclusion and tests a structural pre-tournament signal: hosting **eligibility** from the announced bracket. Eligibility is not an actual-venue label. The model receives no tournament WLoc, city, realized round, scores or winner as a predictor.

Women only, 2016–2019 exploratory folds; 2013 onward training; 12 new small classifiers, four reference replays, zero new rating fits. Keep every upstream kit and checkpoint. Run the terminal tests in START_HERE.md first. Select **Python (March Mania)**.

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
import plotly.io as pio
from IPython.display import display, FileLink
KIT = Path.cwd().resolve()
if not (KIT / 'run_round09.py').is_file():
    KIT = Path.home() / 'march_bracket_context'
assert (KIT / 'run_round09.py').is_file(), 'Open this notebook inside march_bracket_context.'
sys.path.insert(0, str(KIT))
from run_round09 import run_stage
from bracket_plots import figures
pio.renderers.default = 'plotly_mimetype'
print('Kernel:', sys.executable)
print('Kit:', KIT)
print('No AWS API actions, network downloads, Git writes or dependency installations.')

## 1 · Preserve round 08’s result
The proper comparison is **mechanism_given_rates**, not only improvement over the smaller anchor. In the uploaded run it worsened mean Brier by +0.0004202 for men and +0.0005932 for women. Neither primary gate passed. The favorable mean from the men's ordinary-rate control is a separate, unstable observation; its worst season deteriorated by +0.0062071.

In [ ]:
prior = pd.read_csv(KIT / 'evidence/round08/ablations.csv')
display(prior.query("comparison == 'mechanism_given_rates'").round(7))
print(json.dumps(json.loads((KIT / 'evidence/round08/decisions.json').read_text()), indent=2))

## 2 · Definitions and information availability
**Control:** difference in top-four seed status. This ordinary nonlinear seed transform is not a new domain discovery.

**Candidate 1:** the signed top-four status difference, active only when both teams share the same announced regional four-team pod and the top-16-hosting policy applies (2015–2019).

**Candidate 2:** candidate 1 scaled by `10 / sqrt(margin_sd_A**2 + margin_sd_B**2 + 1)`, using cached regular-season margin variability. This is a hypothesis about probability sensitivity, not calibrated uncertainty.

Pods are {1,16,8,9}, {4,13,5,12}, {3,14,6,11}, {2,15,7,10}, separately within each region. Construct all 2,016 potential pairs per season **before** attaching tournament targets.

Known counterexamples prevent an actual-home claim: Stanford could not host in 2017; South Carolina hosted at Charlotte in 2019. Those exceptions are documented in RESEARCH_PLAN.md but are not silently patched using outcome files. Before 2015 the policy feature is zero as inapplicable—not proof that all older games were neutral. Later years/play-in formats are unsupported and must stop.

Preparation verifies seven base snapshots, builds seven small all-pairs tables, and replays four references. **No new model fit in preparation.** Hard limit: 180 seconds.

In [ ]:
run_stage('prepare', max_seconds=180)
RUN = Path(json.loads((KIT / 'reports/latest_run.json').read_text())['run_dir'])
print(json.dumps(json.loads((RUN / 'prepare.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'coverage.csv'))
display(pd.read_csv(RUN / 'prior_replay.csv').round(7))

In [ ]:
registry = pd.read_csv(RUN / 'feature_registry.csv')
display(registry.loc[registry.family != 'anchor', ['feature','family','definition','available_asof']])
print('Keep formulas fixed after seeing the scores. No actual tournament location is inferred.')

## 3 · Fixed four-season experiment
| Recipe | Inputs | Role |
|---|---:|---|
| anchor | 16 | Existing reference replay |
| seed_status | 17 | Nonlinear seed-status control |
| host_context | 18 | Control plus eligibility |
| host_context_scaled | 19 | Control plus both eligibility features |

Primary: **host_context_scaled minus seed_status**. Secondary contrasts isolate the raw eligibility and the variability-scaled addition. Removing a feature does not select a replacement.

C=0.1, mirrored orientations, physical-game weighting and train-only RMS scaling remain unchanged. Fit cap: 12 classifiers. There are four validation folds, always trained on strictly earlier seasons from 2013. The 2016 fit has only one policy-era training season, which limits its interpretation. Hard limit: 180 seconds.

In [ ]:
run_stage('evaluate', max_seconds=180)
metrics = pd.read_csv(RUN / 'metrics.csv')
display(metrics[['Season','recipe','games','brier','log_loss','delta_vs_anchor','source']].round(7))
display(pd.read_csv(RUN / 'ablations.csv').round(7))

## 4 · Check stability, not just the smallest score
A mean primary delta ≤ −0.0005, improvement in at least three of four seasons, and a worst deterioration ≤ +0.003 trigger **CONSIDER_CONFIRMED_VENUE_REPLICATION**. Otherwise stop automatic expansion. This is a compute-allocation rule, not significance, causal attribution or promotion.

All these seasons have been examined before; none is called a fresh holdout. The eligibility proxy must later face historically verified venue exceptions, later-era testing and a fixed production recipe before any submission claim.

In [ ]:
print(json.dumps(json.loads((RUN / 'decisions.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'support_by_fold.csv'))
print('Negative delta favors inclusion; process COMPLETE does not mean the hypothesis helped.')

## 5 · Ten interactive Plotly charts
Prior negative results, announced-bracket context, pair support, Brier, incremental effects, cohort diagnostics, calibration, coefficients and training-only overlap. Sparse calibration and cohort estimates are descriptive.

In [ ]:
plots = figures(RUN, KIT / 'evidence/round08')
assert len(plots) == 10
for fig in plots[:5]:
    fig.show()

In [ ]:
for fig in plots[5:]:
    fig.show()

## 6 · Save and return
Report hard limit: 120 seconds. The small return archive excludes raw rows, fitted models, all-pairs tables, game-level predictions and private notebooks. The self-contained HTML report stays beside the run artifacts. Save this notebook with Ctrl+S and return **reports/milestone_09_return.zip**. Stop this milestone here.

In [ ]:
run_stage('report', max_seconds=120)
record = json.loads((KIT / 'reports/latest_report.json').read_text())
print('Return:', record['return_zip'])
print('Interactive report:', record['html'])
display(FileLink(str(Path(record['return_zip']).relative_to(KIT))))
display(FileLink(str(Path(record['html']).relative_to(KIT))))
print('All successful work remains in the fingerprinted private_runs directory.')